In [9]:
pip install -q rdflib pyvis

Note: you may need to restart the kernel to use updated packages.


In [11]:
import sys; sys.path.append("/home/jovyan/work")
from commun.spark_utils import RAW

import pandas as pd
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD

PARIS =  Namespace("http://example.org/comptage-multimodal#") # noms du notre projet
g = Graph()
g.bind("", PARIS); g.bind("owl", OWL); g.bind("rdfs", RDFS)


g.parse(
    "/home/jovyan/work/semantique/ontologie_comptagesparis.ttl",
    format="turtle"
)

print("Nombre de triples :", len(g))

Nombre de triples : 46


In [12]:

requete = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?classe
WHERE {
    ?classe a owl:Class .
}
"""

resultats = g.query(requete)

for resultat in resultats:
    print(resultat.classe)


http://example.org/comptage-multimodal#ObservationComptage
http://example.org/comptage-multimodal#SiteComptage
http://example.org/comptage-multimodal#ModeDeplacement
http://example.org/comptage-multimodal#TypeVoie


In [13]:

requete = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?propriete
WHERE {
    ?propriete a owl:ObjectProperty .
}
"""

resultats = g.query(requete)

for resultat in resultats:
    print(resultat.propriete)


http://example.org/comptage-multimodal#concerneSite
http://example.org/comptage-multimodal#concerneMode
http://example.org/comptage-multimodal#concerneTypeVoie


In [36]:
#requete 1
PREFIXES = """
PREFIX : <http://example.org/comptage-multimodal#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?mode
WHERE {
    ?observation rdf:type :ObservationComptage .
    ?observation :concerneMode ?mode .
}
ORDER BY ?mode
"""
resultats = g.query(PREFIXES)

for resultat in resultats:
    print(resultat)


In [ ]:



# Créer un graphe de test et y charger l'ontologie
g_test = Graph()
g_test.parse("/home/jovyan/work/semantique/ontologie_comptagesparis.ttl",
             format="turtle")

PARIS = Namespace("http://example.org/comptage-multimodal#")

# Ajouter deux modes de déplacement
g_test.add((
    PARIS.observation_test_1,
    RDF.type,
    PARIS.ObservationComptage
))
g_test.add((
    PARIS.observation_test_1,
    PARIS.concerneMode,
    PARIS.mode_trottinette
))
g_test.add((
    PARIS.observation_test_2,
    RDF.type,
    PARIS.ObservationComptage
))
g_test.add((
    PARIS.observation_test_2,
    PARIS.concerneMode,
    PARIS.mode_velo
))

# Ajouter les noms des modes pour les afficher
g_test.add((
    PARIS.mode_trottinette,
    RDFS.label,
    Literal("Trottinettes", lang="fr")
))
g_test.add((
    PARIS.mode_velo,
    RDFS.label,
    Literal("Vélos", lang="fr")
))

print("Nombre de triples dans le graphe de test :", len(g_test))


FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/work/semantique/ontology/ontologie.ttl'

In [ ]:
#requete 2 -- recherche les sites de comptages,recuperer les id & nom tout en supprimant les doublons
PREFIXES = """
PREFIX : <http://example.org/comptage-multimodal#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?identifiant ?nom
WHERE {
    ?site rdf:type :SiteComptage .
    ?site :identifiantSite ?identifiant .
    ?site :nomSite ?nom .
}
ORDER BY ?nom
"""


In [22]:
#requete 3-- recupere les obs,date & nb de vehicle et filtre par jan2025 in asc order
PREFIXES = """
PREFIX : <http://example.org/comptage-multimodal#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?observation ?date ?nombre
WHERE {
    ?observation rdf:type :ObservationComptage .
    ?observation :dateHeureComptage ?date .
    ?observation :nombreVehicules ?nombre .

    FILTER (
        ?date >= "2025-01-01T00:00:00"^^xsd:dateTime &&
        ?date < "2025-02-01T00:00:00"^^xsd:dateTime
    )
}
ORDER BY ?date
"""

In [29]:

from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, XSD

# Namespace de ton ontologie
PARIS = Namespace("http://example.org/comptage-multimodal#")

# Charger l'ontologie dans un nouveau graphe
g_test = Graph()
g_test.parse(
    "/home/jovyan/work/semantique/ontologie_comptagesparis.ttl",
    format="turtle"
)

# Transformer les 5 lignes du CSV en triples RDF
for index, ligne in df.iterrows():
    observation = URIRef(f"http://example.org/observation/{index}")

    mode = str(ligne["Mode déplacement"]).strip()
    site_id = str(ligne["Identifiant site"]).strip()
    nom_site = str(ligne["Nom du site"]).strip()

    g_test.add((observation, RDF.type, PARIS.ObservationComptage))

    mode_uri = URIRef(
        "http://example.org/mode/" + str(index)
    )
    g_test.add((mode_uri, RDF.type, PARIS.ModeDeplacement))
    g_test.add((mode_uri, PARIS.nomSite, Literal(mode)))

    g_test.add((observation, PARIS.concerneMode, mode_uri))

print("Nombre de triples dans le graphe de test :", len(g_test))


Nombre de triples dans le graphe de test : 66


In [34]:

requete_modes = """
PREFIX : <http://example.org/comptage-multimodal#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?nomMode
WHERE {
    ?observation rdf:type :ObservationComptage .
    ?observation :concerneMode ?mode .
    ?mode :nomSite ?nomMode .
}
"""

for resultat in g_test.query(requete_modes):
    print(resultat.nomMode)


Trottinettes + vélos
van
Véhicules légers < 3,5t
Trottinettes
